# SUPER SENTAI ALL-IN-ONE PIPELINE (3D CGI TOKUSATSU)
Notebook duy nhất chạy tuần tự từ **A → Z** trên **Google Colab GPU** hoặc **IDE Colab Extension**:
1. Cài đặt thư viện & kiểm tra GPU.
2. Khởi tạo vũ trụ (`init_sentai_universe.py`).
3. Tạo toàn bộ bối cảnh & nhân vật **3D CGI** + tách nền `rembg` (`fetch_ai_characters.py --force`).
4. Tạo **12 Stock Clips đánh đấm, biến thân & Robot 3D CGI** (`generate_stock_clips.py` — `AnimateDiff + DreamShaper-8`).
5. Tạo **Diễn xuất cử động khuôn mặt** (`LivePortrait`) lưu vào `assets/characters/<char_id>/liveportrait_civilian.mp4`.
6. Tạo **Giọng lồng tiếng đa nhân vật** (`F5-TTS` từ `assets/voices/<char_id>.wav`).
7. Render phim hoàn chỉnh (`sentai_series_engine.py`) & tạo link **Gradio** tải phim + `colab_assets.zip`.

In [ ]:
# BƯỚC 0: CẤU HÌNH CHUNG (CHỈ SỬA Ở ĐÂY)
import os
from pathlib import Path

PROJECT_DIR = Path(".")
EP_FILE = "episodes/ep_001.json"
MODE = "youtube"  # "youtube" (16:9) hoặc "tiktok" (9:16)

# Bật/tắt từng bước theo nhu cầu
RUN_INIT_UNIVERSE = True
RUN_FETCH_AI_CHARS = True
RUN_STOCK_CLIPS = True
RUN_LIVEPORTRAIT = True
RUN_F5TTS_VOICE = False   # Tắt mặc định vì F5TTS_v1_Base là model Anh/Trung (dùng edge-tts + pitch_scale chuẩn tiếng Việt)
REF_VOICE_WAV = "ref_voice.wav"
RUN_FINAL_RENDER = True

# Kiểm tra môi trường Colab
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    pass

print(f"IN_COLAB={IN_COLAB} | EP_FILE={EP_FILE} | MODE={MODE}")

In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN, FONT TIẾNG VIỆT & KIỂM TRA GPU
!apt-get -qq update && apt-get -qq install -y fonts-dejavu-core
!pip install -q edge-tts gTTS "Pillow<12.0.0" numpy pysubs2 mutagen requests imageio-ffmpeg rembg onnxruntime pydub
!pip install -q diffusers transformers accelerate sentencepiece

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ĐỒNG BỘ CODE TỪ GITHUB KHI CHẠY TRÊN COLAB
REPO_URL = "https://github.com/nguyenkhanhan123/Test.git"

if IN_COLAB and not Path("sentai_series_engine.py").exists():
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not Path(repo_name).exists():
        !git clone {REPO_URL}
    %cd {repo_name}
    !git pull
elif IN_COLAB and Path(".git").exists():
    !git pull

In [ ]:
# BƯỚC 2: KHỞI TẠO VŨ TRỤ & TÀI NGUYÊN CỐ ĐỊNH
if RUN_INIT_UNIVERSE:
    !python init_sentai_universe.py

In [ ]:
# BƯỚC 3: TẠO TOÀN BỘ BỐI CẢNH & ẢNH NHÂN VẬT 3D CGI BẰNG GPU
if RUN_FETCH_AI_CHARS:
    !python fetch_ai_characters.py --force

In [ ]:
# BƯỚC 4: TẠO 12 CLIP ĐÁNH ĐẤM, BIẾN THÂN & ROBOT 3D CGI BẰNG LTX-VIDEO (TỐI ƯU VRAM T4 15GB)
if RUN_STOCK_CLIPS:
    !python -u generate_stock_clips.py

In [ ]:
# BƯỚC 5: TẠO DIỄN XUẤT KHUÔN MẶT BẰNG LIVEPORTRAIT (LƯU VÀO assets/characters/<char_id>/liveportrait_civilian.mp4)
if RUN_LIVEPORTRAIT:
    !python -u run_liveportrait.py

In [ ]:
# BƯỚC 6: TẠO GIỌNG LỒNG TIẾNG CẢM XÚC ĐA NHÂN VẬT BẰNG F5-TTS (ĐẶT FILE MẪU 5s VÀO assets/voices/<char_id>.wav)
if RUN_F5TTS_VOICE:
    import json, subprocess
    from pydub import AudioSegment

    voices_dir = Path("assets/voices")
    voices_dir.mkdir(parents=True, exist_ok=True)
    ep = json.load(open(EP_FILE, encoding="utf-8"))
    ep_code = f"ep_{ep['episode_number']:03d}"
    out_dir = Path("build_output") / ep_code / "audio"
    out_dir.mkdir(parents=True, exist_ok=True)
    ref_text = "Đây là giọng mẫu chuẩn dùng để lồng tiếng cho chiến binh Ngũ Hành Chiến Đội."

    for act in ep["acts"]:
        aid = act["act_id"]
        for idx, sc in enumerate(act["scenes"]):
            sid = f"{aid}_s{idx:03d}"
            cid = sc.get("char_id") or sc.get("speaker", "hoa_do")
            ref_wav = next(
                (p for p in [voices_dir / f"{cid}.wav", voices_dir / "default.wav", Path(REF_VOICE_WAV)] if p.exists()),
                None,
            )
            if ref_wav is None:
                continue
            wav_out = out_dir / f"{sid}.wav"
            mp3_out = out_dir / f"{sid}.mp3"
            print(f"F5-TTS [{cid} -> {ref_wav.name}]:", sid)
            subprocess.run([
                "f5-tts_infer-cli",
                "--model", "F5TTS_v1_Base",
                "--ref_audio", str(ref_wav),
                "--ref_text", ref_text,
                "--gen_text", sc["text"],
                "--output_dir", str(out_dir),
                "--output_file", f"{sid}.wav",
            ], check=True)
            AudioSegment.from_wav(wav_out).export(mp3_out, format="mp3", bitrate="192k")
            wav_out.unlink(missing_ok=True)

In [ ]:
# BƯỚC 7: RENDER PHIM HOÀN CHỈNH (YOUTUBE 16:9 HOẶC TIKTOK 9:16)
if RUN_FINAL_RENDER:
    if not RUN_F5TTS_VOICE:
        !rm -rf build_output/ep_001/audio/*.mp3 build_output/ep_001/audio/*.wav
    !python -u sentai_series_engine.py --ep {EP_FILE} --mode {MODE} --force

In [ ]:
# BƯỚC 8: ĐÓNG GÓI & TẠO LINK TẢI PHIM + TÀI NGUYÊN (GRADIO)
import json
ep = json.load(open(EP_FILE, encoding="utf-8"))
ep_code = f"ep_{ep['episode_number']:03d}"
final_mp4 = Path(f"build_output/NguHanhChienDoi_{ep_code}_{MODE.upper()}_FULL.mp4")
print("File phim:", final_mp4.resolve(), "| Tồn tại:", final_mp4.exists())

if IN_COLAB and final_mp4.exists():
    !zip -r -q colab_assets.zip assets/backgrounds assets/characters assets/stock_clips
    !pip install -q gradio
    import gradio as gr
    with gr.Blocks() as demo:
        gr.File(value=str(final_mp4), label=f"Phim {ep_code.upper()} ({MODE.upper()})")
        gr.File(value="colab_assets.zip", label="Kho ảnh nền, nhân vật, Stock Clips & LivePortrait")
    demo.launch(share=True)